# 4장 1강: 데이터 불균형과 평가지표 왜곡

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 데이터 불균형이 분류 모델 평가에 어떤 영향을 주는지 확인합니다.

- `Class` 비율을 확인하여 불균형 정도를 파악합니다.
- Accuracy만 보면 생길 수 있는 착시를 확인합니다.
- Precision, Recall, F1-score, PR-AUC를 함께 확인합니다.
- `class_weight`, Random Undersampling, SMOTE를 적용하여 성능 변화를 비교합니다.
- threshold 변화에 따른 Precision / Recall 변화를 확인합니다.

> 이 실습에서는 4장 1강 교안에 나온 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습에서는 모델 입력으로 `V1`부터 `V28`까지 사용하고, `Class`를 예측 대상으로 사용합니다.

## 실습 준비

1. 데이터 크기를 확인합니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 X, `Class`를 y로 분리합니다.
4. Train / Test 데이터를 분리합니다.
5. 리샘플링은 **Train 데이터에만** 적용합니다.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

data = pd.read_csv("creditcard.csv")

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(data["Class"].value_counts().to_frame("count"))

print("\n[Class 비율]")
display(data["Class"].value_counts(normalize=True).to_frame("ratio"))

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain 크기:", X_train.shape)
print("Test 크기:", X_test.shape)

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727



Train 크기: (227845, 28)
Test 크기: (56962, 28)


---

# 필수 1. Accuracy의 함정과 불균형 지표 확인

## 배경

Credit Card Fraud 데이터는 정상 거래가 매우 많고 사기 거래가 매우 적은 불균형 데이터입니다.

이런 상황에서는 Accuracy만 보면 모델이 매우 좋아 보일 수 있습니다.

## 문제 1. 모든 거래를 정상이라고 예측해보세요.

### 요구사항

1. Test 데이터의 모든 거래를 `0`으로 예측합니다.
2. Accuracy를 계산합니다.
3. Recall을 계산합니다.
4. Accuracy와 Recall 결과를 비교합니다.

### 확인 포인트

- 모든 거래를 정상이라고 예측해도 Accuracy가 매우 높게 나오는가?
- 사기 거래를 하나도 잡지 못하면 Recall은 어떻게 되는가?

In [2]:
# TODO
# 모든 거래를 정상(0)으로 예측하고 Accuracy / Recall을 계산하세요.
# 1. Test 데이터의 모든 거래를 0으로 예측하는 가상 더미 예측값 생성
y_pred_all_zeros = np.zeros(len(y_test))

# 2. Accuracy 계산
acc_all_zeros = accuracy_score(y_test, y_pred_all_zeros)

# 3. Recall(재현율) 계산 (pos_label=1인 사기 거래 기준)
recall_all_zeros = recall_score(y_test, y_pred_all_zeros, zero_division=0)

# 4. 결과 출력 및 비교
print("=== 모든 거래를 0(정상)으로 단순 예측한 결과 ===")
print(f"Accuracy (정확도) : {acc_all_zeros:.4%}")
print(f"Recall   (재현율) : {recall_all_zeros:.4%}")

# 오차행렬(Confusion Matrix) 확인
cm = confusion_matrix(y_test, y_pred_all_zeros)
cm_df = pd.DataFrame(
    cm,
    index=["실제 정상(0)", "실제 사기(1)"],
    columns=["예측 정상(0)", "예측 사기(1)"]
)
print("\n[Confusion Matrix]")
display(cm_df)

=== 모든 거래를 0(정상)으로 단순 예측한 결과 ===
Accuracy (정확도) : 99.8280%
Recall   (재현율) : 0.0000%

[Confusion Matrix]


,예측 정상(0),예측 사기(1)
실제 정상(0),56864,0
실제 사기(1),98,0


#### 결과
| 실제 정답 | 정상 0으로 예측 | 사기 1로 예측 | 내용 |
|---|---|---|---|
| 정상 0 | TN = 56,864 | FP = 0 | 정상 거래는 전부 맞춤 |
| 사기 1 | FN = 98 | TP = 0 | 사기 거래는 전부 다 틀림|

- TN은 정상을 정상으로 맞춘 건수, TP는 사기를 사기로 맞힌 건수
- FP는 정상인데 사기라고 잘못 예측한 오탐
- FN은 사기인데 정상이라고 잘못 예측한 미탐

### Q1. Accuracy가 높더라도 이 모델을 좋은 사기 탐지 모델이라고 보기 어려운 이유는 무엇인가요?

**답변:**
- 데이터의 대부분이 정상거래이기 때문에 전부 다 정상거래로 예측을 해도 정확도는 높다.
- 하지만 실제 사기 거래를 전혀 걸러내지 못하기 때문에 좋은 모델이라고 볼 수는 없다.
- Recall, Precision등도 같이 확인해야 모델의 성능을 평가할 수 있음
###
- 불균형 데이터에서는 Accuracy가 높아도 소수 클래스를 전혀 탐지 못할 수 있다.

## 문제 2. 기본 Logistic Regression의 여러 지표를 확인하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 학습합니다.
2. Test 데이터를 예측합니다.
3. Accuracy를 계산합니다.
4. Precision을 계산합니다.
5. Recall을 계산합니다.
6. F1-score를 계산합니다.
7. `predict_proba()`의 양성 클래스 확률로 PR-AUC를 계산합니다.
8. 혼동행렬을 확인합니다.

### 확인 포인트

- Accuracy 하나만 보는 것과 여러 지표를 함께 보는 것의 차이를 이해했는가?
- Precision과 Recall이 각각 어떤 오류와 연결되는지 설명할 수 있는가?
- PR-AUC가 불균형 데이터에서 왜 중요한지 이해했는가?

In [3]:
# TODO
# 기본 Logistic Regression의 Accuracy / Precision / Recall / F1 / PR-AUC / 혼동행렬을 확인하세요.
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# 1. LogisticRegression 모델 생성 및 Train 데이터 학습
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 2. Test 데이터 예측 (이진 레이블 및 양성 확률)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # 사기 거래(Class=1) 확률

# 3~7. 각 평가지표 계산
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
pr_auc = average_precision_score(y_test, y_prob)

# 8. 혼동행렬(Confusion Matrix) 계산 및 DataFrame 정리
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["실제 정상(0)", "실제 사기(1)"],
    columns=["예측 정상(0)", "예측 사기(1)"]
)

# 결과 출력
print("=== 기본 Logistic Regression 평가 결과 ===")
print(f"1. Accuracy  (정확도) : {acc:.4%}")
print(f"2. Precision (정밀도) : {prec:.4f} ({prec*100:.2f}%)")
print(f"3. Recall    (재현율) : {rec:.4f} ({rec*100:.2f}%)")
print(f"4. F1-Score  (F1점수) : {f1:.4f}")
print(f"5. PR-AUC    (평균정밀도): {pr_auc:.4f}")

print("\n=== 혼동행렬 (Confusion Matrix) ===")
display(cm_df)

=== 기본 Logistic Regression 평가 결과 ===
1. Accuracy  (정확도) : 99.9157%
2. Precision (정밀도) : 0.8289 (82.89%)
3. Recall    (재현율) : 0.6429 (64.29%)
4. F1-Score  (F1점수) : 0.7241
5. PR-AUC    (평균정밀도): 0.7436

=== 혼동행렬 (Confusion Matrix) ===


,예측 정상(0),예측 사기(1)
실제 정상(0),56851,13
실제 사기(1),35,63


### Q2. 사기 거래를 놓치는 FN의 비용이 크다면 Precision과 Recall 중 무엇을 더 중요하게 봐야 하나요?

#### 혼동행렬
| 실제 정답 / 예측 | 정상 0 | 사기 | 실제 거래 합계 |
|---|---|---|---|
| 정상 0 | TN = 56,851 | FP = 13 | 56,864 |
| 사기 1| FN = 35 | TP = 63 | 98 |
| 예측 합계 | 56,886 | 76 | 56,962 |

- TP : 사기 63건을 사기라고 올바르게 탐지한 경우
- FN : 사기 35건을 정상이라고 잘못 예측한 경우
- FP : 정상 13건을 사기라고 잘못 예측한 경우
- TN : 정상 56,851건을 정상이라고 올바르게 예측한 경우
###
- 사기라고 예측한 거래는 TP + FP = 76건
- 실제 사기 거래는 TP + FN = 98건
###
|지표|계산식|결과|내용|
|---|---|---|---|
|Accuracy|(56,851 + 63) / 56.962 | 0.999 | 전체 거래의 약 99.9%를 올바르게 분류 |
|Precision|63 / (63 + 13)| 0.828| 사기라고 예측한 76건 중 약 82.8%가 실제 사기|
|Recall| 63 / (63 + 35)| 0.642| 실제 사기 98건 중 64.2%정도를 올바르게 분류|
|F1| 2 * 63 / (2 * 63 + 13 + 35)| 0.724| Precision과 Recall의 조화평균|
|PR-AUC| 정답과 사기 점수| 0.743| Precision-Recall 성능 요약|

- Precision은 사기라고 말한 것을 얼마나 믿을 수 있는가?
- Recall은 사기를 실제로 얼마나 놓치지 않고 찾았는지?
###
- 종합 해석: 모두 정상으로 예측한 기준선은 0건을 탐지해냄, 기본 모델같은 경우 63건을 찾고 35건을 놓쳤으며 정상 13건을 사기로 잘못 예측함.
- 사기를 사기로 탐지하는 능력은 있고 정확도도 매우 높은 편이지만 Recall이 64%정도이기 때문에 사기를 거의 다 잡아낼 수 있는 모델이라고 평가할 수는 없다.

**답변:**  
- Recall: 실제 사기중에서 얼마나 사기를 정확하게 찾아냈는지를 나타내기 때문
- 사기를 놓치는 FN을 중요하게 생각한다면 Recall 값을 최대한 높여서 만들어야함

---

# 필수 2. 불균형 처리 기법 비교

## 배경

교안에서는 불균형 처리 방법으로 다음 세 가지를 다룹니다.

- `class_weight`
- Random Undersampling
- SMOTE

이번 문제에서는 세 방법을 Train 데이터에만 적용한 뒤 같은 Test 데이터에서 성능을 비교합니다.

## 문제 1. class_weight를 적용하세요.

### 요구사항

1. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
2. Train 데이터로 학습합니다.
3. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스에 더 큰 가중치를 주면 Recall이 어떻게 변하는가?
- Precision과 Recall이 동시에 항상 좋아지는 것은 아니라는 점을 확인했는가?

In [4]:
# TODO
# class_weight="balanced" 모델을 학습하고 지표를 확인하세요.
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# 1. class_weight="balanced" 옵션을 적용한 Logistic Regression 모델 생성
model_balanced = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)

# 2. Train 데이터로 학습
model_balanced.fit(X_train, y_train)

# 3. Test 데이터 예측
y_pred_bal = model_balanced.predict(X_test)
y_prob_bal = model_balanced.predict_proba(X_test)[:, 1]

# 4. 평가지표 계산
prec_bal = precision_score(y_test, y_pred_bal, zero_division=0)
rec_bal = recall_score(y_test, y_pred_bal, zero_division=0)
f1_bal = f1_score(y_test, y_pred_bal, zero_division=0)
pr_auc_bal = average_precision_score(y_test, y_prob_bal)

print("=== class_weight='balanced' 평가 결과 ===")
print(f"Precision (정밀도) : {prec_bal:.4f} ({prec_bal*100:.2f}%)")
print(f"Recall    (재현율) : {rec_bal:.4f} ({rec_bal*100:.2f}%)")
print(f"F1-Score  (F1점수) : {f1_bal:.4f}")
print(f"PR-AUC    (평균정밀도): {pr_auc_bal:.4f}")

# 오차행렬 확인
cm_bal = confusion_matrix(y_test, y_pred_bal)
cm_bal_df = pd.DataFrame(
    cm_bal,
    index=["실제 정상(0)", "실제 사기(1)"],
    columns=["예측 정상(0)", "예측 사기(1)"]
)
print("\n[Confusion Matrix]")
display(cm_bal_df)

=== class_weight='balanced' 평가 결과 ===
Precision (정밀도) : 0.0579 (5.79%)
Recall    (재현율) : 0.9184 (91.84%)
F1-Score  (F1점수) : 0.1089
PR-AUC    (평균정밀도): 0.7092

[Confusion Matrix]


,예측 정상(0),예측 사기(1)
실제 정상(0),55399,1465
실제 사기(1),8,90


- balanced는 어떤 가중치를 부여하는가?
- 클래스 K의 학습 표본 수가 NK, 전체 학습 표본수가 N, 클래스 수가 K이면 
- $$w_k \frac{n}{k n_k}$$
###
- 가중치가 0.50정도 사기 한건이 약 289.14 정도의 수치
- 같은 정도의 손실이라면 사기 한건이 정상 한건보다 약 577.3배 큰 비중을 가진다.
###
- Precision 처음에 82% -> 지금 5% -> 급격하게 사기라고 예측한 것중에 실제 사기였던 비율이 크게 낮아짐
- Recall 처음에 64% -> 지금 91% -> 실제 사기를 기존보다 더 잘 예측
- F1 처음에 72% -> 지금 10% -> Precision의 값이 매우 작아 종합 점수가 매우 떨어짐
###
- TP : 기존 모델 63건, 클래스 가중치 부여한 경우 90건 -> 사기 사건 27건 추가 탐지
- FN : 기존 모델 35건, 클레스 가중치 부여한 경우 8건 -> 사기라고 탐지 못한 경우 27건 감소
- FP : 기존 모델 13건, 클래스 가중치 부여한 경우 1,465건 -> 오탐 1,452건 증가
- TP + FP : 기존 모델 76건, 클래스 가중치 부여한 경우 1,555 -> 대부분 사기라고 예측을 해버림

###
- 해석: 모두 정상으로 예측한 기준선은 사기를 0건 찾았지만, 기본 모델은 63건을 찾고 35건을 놓쳤고 정상 13건을 오탐했다.
- 사기 탐지 능력은 있으나 Accuracy가 90%를 넘는다고 매우 좋은 모델이다라고 해석할 수는 없다. 실제 사기의 약 35%를 놓치기 때문이다.

## 문제 2. Random Undersampling을 적용하세요.

### 요구사항

1. `RandomUnderSampler(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 다수 클래스 샘플이 줄어드는가?
- Test 데이터에는 Undersampling을 적용하지 않았는가?

In [5]:
# TODO
# Random Undersampling을 Train 데이터에만 적용하세요.
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# 1. RandomUnderSampler 객체 생성
rus = RandomUnderSampler(random_state=42)

# 2. Train 데이터에만 fit_resample() 적용 (Test 데이터는 원본 유지)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

# 3. 리샘플링 전후 Train 데이터 클래스 개수 확인
print("=== 리샘플링 전 Train Class 분포 ===")
display(y_train.value_counts().to_frame("Before_RUS"))

print("\n=== 리샘플링 후 Train Class 분포 ===")
display(y_train_rus.value_counts().to_frame("After_RUS"))

# 4. Logistic Regression 모델 학습
model_rus = LogisticRegression(max_iter=1000, random_state=42)
model_rus.fit(X_train_rus, y_train_rus)

# 5. Test 데이터(불균형 상태 그대로) 예측 및 지표 계산
y_pred_rus = model_rus.predict(X_test)
y_prob_rus = model_rus.predict_proba(X_test)[:, 1]

prec_rus = precision_score(y_test, y_pred_rus, zero_division=0)
rec_rus = recall_score(y_test, y_pred_rus, zero_division=0)
f1_rus = f1_score(y_test, y_pred_rus, zero_division=0)
pr_auc_rus = average_precision_score(y_test, y_prob_rus)

print("\n=== Random Undersampling 모델 평가 결과 ===")
print(f"Precision (정밀도) : {prec_rus:.4f} ({prec_rus*100:.2f}%)")
print(f"Recall    (재현율) : {rec_rus:.4f} ({rec_rus*100:.2f}%)")
print(f"F1-Score  (F1점수) : {f1_rus:.4f}")
print(f"PR-AUC    (평균정밀도): {pr_auc_rus:.4f}")

# 오차행렬 확인
cm_rus = confusion_matrix(y_test, y_pred_rus)
cm_rus_df = pd.DataFrame(
    cm_rus,
    index=["실제 정상(0)", "실제 사기(1)"],
    columns=["예측 정상(0)", "예측 사기(1)"]
)
print("\n[Confusion Matrix]")
display(cm_rus_df)

=== 리샘플링 전 Train Class 분포 ===


,Before_RUS
Class,
0,227451
1,394



=== 리샘플링 후 Train Class 분포 ===


,After_RUS
Class,
0,394
1,394



=== Random Undersampling 모델 평가 결과 ===
Precision (정밀도) : 0.0376 (3.76%)
Recall    (재현율) : 0.9184 (91.84%)
F1-Score  (F1점수) : 0.0722
PR-AUC    (평균정밀도): 0.6358

[Confusion Matrix]


,예측 정상(0),예측 사기(1)
실제 정상(0),54560,2304
실제 사기(1),8,90


### 왜 언더 샘플링 방법을 사용하는가?
- 다수 클래스의 수를 줄이면 학습 자료에서 사기 표본의 상대적 비중이 높아짐
- 처리할 자료의 양이 줄어서 학습 속도 향상을 기대할 수 있음
- 다만, 현재 무작위로 남긴 394건이 정상 거래의 다양한 패턴을 충분히 반영하지 못하는 문제가 발생할 수 있음
### 
- 샘플링 비율, 남기는 표본, 모델 설정을 다르게 하면 모델의 결과가 달라질 수 있다.
- 이번 언더샘플링은 사기 탐지(Recall)의 값은 높았지만 오탐이 매우 많이 발생했다.
- 언더샘플링 자체가 잘못된 것이 아니라 데이터셋이 언더샘플링에 부적절했다고 판단한다.

## 문제 3. SMOTE를 적용하세요.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스 샘플이 증가하는가?
- SMOTE가 Test 데이터에 적용되지 않았는가?
- Recall 개선과 Precision 감소가 함께 나타날 수 있음을 확인했는가?

In [6]:
# TODO
# SMOTE를 Train 데이터에만 적용하세요.
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# 1. SMOTE 객체 생성
smote = SMOTE(random_state=42)

# 2. Train 데이터에만 fit_resample() 적용 (소수 클래스 오버샘플링)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 3. 리샘플링 전후 Train 데이터 클래스 개수 확인
print("=== 리샘플링 전 Train Class 분포 ===")
display(y_train.value_counts().to_frame("Before_SMOTE"))

print("\n=== 리샘플링 후 Train Class 분포 ===")
display(y_train_smote.value_counts().to_frame("After_SMOTE"))

# 4. Logistic Regression 모델 학습
model_smote = LogisticRegression(max_iter=1000, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

# 5. Test 데이터(불균형 상태 그대로) 예측 및 지표 계산
y_pred_smote = model_smote.predict(X_test)
y_prob_smote = model_smote.predict_proba(X_test)[:, 1]

prec_smote = precision_score(y_test, y_pred_smote, zero_division=0)
rec_smote = recall_score(y_test, y_pred_smote, zero_division=0)
f1_smote = f1_score(y_test, y_pred_smote, zero_division=0)
pr_auc_smote = average_precision_score(y_test, y_prob_smote)

print("\n=== SMOTE 모델 평가 결과 ===")
print(f"Precision (정밀도) : {prec_smote:.4f} ({prec_smote*100:.2f}%)")
print(f"Recall    (재현율) : {rec_smote:.4f} ({rec_smote*100:.2f}%)")
print(f"F1-Score  (F1점수) : {f1_smote:.4f}")
print(f"PR-AUC    (평균정밀도): {pr_auc_smote:.4f}")

# 오차행렬 확인
cm_smote = confusion_matrix(y_test, y_pred_smote)
cm_smote_df = pd.DataFrame(
    cm_smote,
    index=["실제 정상(0)", "실제 사기(1)"],
    columns=["예측 정상(0)", "예측 사기(1)"]
)
print("\n[Confusion Matrix]")
display(cm_smote_df)

=== 리샘플링 전 Train Class 분포 ===


,Before_SMOTE
Class,
0,227451
1,394



=== 리샘플링 후 Train Class 분포 ===


,After_SMOTE
Class,
0,227451
1,227451



=== SMOTE 모델 평가 결과 ===
Precision (정밀도) : 0.0556 (5.56%)
Recall    (재현율) : 0.9184 (91.84%)
F1-Score  (F1점수) : 0.1048
PR-AUC    (평균정밀도): 0.7188

[Confusion Matrix]


,예측 정상(0),예측 사기(1)
실제 정상(0),55334,1530
실제 사기(1),8,90


### Q3. 리샘플링을 Test 데이터에도 적용하면 안 되는 이유는 무엇인가요?

**답변:**  
- Test 데이터는 실제 배포 환경과 유사한 상황을 가정하고 실시간으로 들어오는 데이터들이 많다.
- 그런데 인위적으로 리샘플링을 진행한다면 기존에 실제 환경과 다른 데이터셋들이 발생할 수 있다. 과적합이 발생할 가능성이 있다.
###
- Undersampling과 SMOTE는 반드시 Train 데이터에만 적용한다.

### SMOTE
- 소수 클래스의 기존 표본과 가까운 클래스 이웃을 합성하여 표본을 만든다. 
- -> 사기행을 중복하여 생성하는 방식 X, 사기 394건을 참고하여 합성한 학습자료
###
- 가중치, 언더샘플링 SMOTE 비교

|처리|TP|FN|FP|전체 사기 예측|
|---|---|---|---|---|
|Class Weight|90|8|1,465|1,555|
|Random Undersampling|90|8|2,304|2,394|
|SMOTE|90|8|1,530|1,620|

- Class Weight가 SMOTE보다 오탐이 65건 더 적었고, AP는 SMOTE가 약 0.718로 Class Weight보다 약 0.09 높다.
###
- SMOTE의 한계점
- 가까운 사기 표본 사이의 실제 사기 분포를 잘 나타낸다면 도움이 될 가능성이 높음
- 그러나 이상치 주변이나 경계가 모호한 경우 의도치 않은 합성 표본이 발생할 수 있음
- 합성 표본을 많이 만드는 것이 모든 성능평가 지표의 상승을 의미하지는 않음

---

# 과제 1. 불균형 처리 전후 성능과 threshold 비교

## 배경

필수 실습에서는 기본 모델과 세 가지 불균형 처리 방법을 확인했습니다.

이번 과제에서는 **기본 Logistic Regression과 class_weight 모델**을 비교하고, class_weight 모델에서 threshold에 따른 Precision / Recall 변화를 확인합니다.

> 과제는 필수 실습에서 사용한 지표와 방법을 그대로 비교하는 수준입니다.

## 요구사항

1. 기본 Logistic Regression과 `class_weight="balanced"` 모델의 다음 지표를 하나의 DataFrame으로 정리합니다.
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - PR-AUC
2. 두 모델의 Recall과 Precision 차이를 설명합니다.
3. `class_weight="balanced"` 모델의 양성 확률을 사용합니다.
4. threshold를 `0.3`, `0.5`, `0.7`로 바꾸어 예측합니다.
5. 각 threshold에서 Precision과 Recall을 계산합니다.
6. threshold 변화에 따라 두 지표가 어떻게 달라지는지 설명합니다.
7. 사기 거래를 놓치는 FN의 비용이 크다고 가정할 때, threshold를 낮추는 방향이 어떤 의미인지 설명합니다.

8. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만 사용할 때의 문제를 설명합니다.
9. FN의 비용과 데이터 특성을 고려해 `class_weight`, Random Undersampling, SMOTE 중 우선 적용할 방법을 하나 선택하고 이유와 한계를 설명합니다. 추가 모델 구현은 요구하지 않습니다.
10. Q4~Q7에 모두 답합니다.

> 처리 기법 선택은 근거 있는 제안으로 작성합니다. Test 결과를 보고 처리 기법이나 threshold를 반복 조정하지 않으며, 실제 설정 선택에는 Train 내부 검증 또는 교차검증이 필요합니다.

### 확인 포인트

- 불균형 처리 전후를 Accuracy 하나가 아니라 여러 지표로 비교했는가?
- threshold를 낮추면 Recall이 높아지는 경향을 설명할 수 있는가?
- threshold를 높이면 Precision이 높아지는 경향을 설명할 수 있는가?
- 단일한 정답 threshold가 있는 것이 아니라 비용 구조에 따라 결정해야 한다는 점을 이해했는가?
- 클래스 비율 때문에 Accuracy가 사기 탐지 성능을 오해하게 할 수 있음을 설명했는가?
- FN 비용과 데이터 특성을 근거로 처리 기법의 선택 이유·한계를 제시했는가?


In [8]:
# TODO
# 기본 모델 / class_weight 모델 비교와 threshold 0.3, 0.5, 0.7 비교를 진행하세요.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    auc
)

# ==============================================================================
# 1 & 2. 기본 Logistic Regression vs class_weight='balanced' 모델 비교
# ==============================================================================
models = {
    "기본 모델 (Default)": LogisticRegression(max_iter=1000, random_state=42),
    "가중치 조정 (class_weight='balanced')": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
}

model_metrics = []

for name, clf in models.items():
    # 훈련 데이터 학습 및 테스트 데이터 예측
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    
    # 지표 계산
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # PR-AUC 계산
    p_curve, r_curve, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(r_curve, p_curve)
    
    model_metrics.append({
        "모델": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "PR-AUC": pr_auc
    })

comparison_df = pd.DataFrame(model_metrics)

print("=== [1] 기본 모델 vs class_weight='balanced' 성능 비교 ===")
display(comparison_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-score": "{:.4f}",
    "PR-AUC": "{:.4f}"
}))

# ==============================================================================
# 3 ~ 5. class_weight='balanced' 모델의 Threshold별 Precision / Recall 계산
# ==============================================================================
# class_weight='balanced' 모델의 양성(Class 1, 사기) 예측 확률
balanced_model = models["가중치 조정 (class_weight='balanced')"]
y_prob_balanced = balanced_model.predict_proba(X_test)[:, 1]

thresholds = [0.3, 0.5, 0.7]
threshold_metrics = []

for th in thresholds:
    # 임계값에 따른 이진 예측값 변환
    y_pred_th = (y_prob_balanced >= th).astype(int)
    
    prec_th = precision_score(y_test, y_pred_th, zero_division=0)
    rec_th = recall_score(y_test, y_pred_th)
    f1_th = f1_score(y_test, y_pred_th)
    
    threshold_metrics.append({
        "Threshold": th,
        "Precision": prec_th,
        "Recall": rec_th,
        "F1-score": f1_th
    })

threshold_df = pd.DataFrame(threshold_metrics)

print("\n=== [2] Threshold에 따른 Precision 및 Recall 변화 ===")
display(threshold_df.style.format({
    "Threshold": "{:.1f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-score": "{:.4f}"
}))

=== [1] 기본 모델 vs class_weight='balanced' 성능 비교 ===


,모델,Accuracy,Precision,Recall,F1-score,PR-AUC
0,기본 모델 (Default),0.9992,0.8289,0.6429,0.7241,0.7424
1,가중치 조정 (class_weight='balanced'),0.9741,0.0579,0.9184,0.1089,0.7533



=== [2] Threshold에 따른 Precision 및 Recall 변화 ===


,Threshold,Precision,Recall,F1-score
0,0.3,0.0259,0.9184,0.0503
1,0.5,0.0579,0.9184,0.1089
2,0.7,0.1146,0.8980,0.2032


#### Q4. 사기 거래를 놓치는 비용이 매우 크다면 threshold를 낮추는 방향은 어떤 장점과 단점이 있나요?

**답변:**  
- threshold를 낮추면 사기 거래로 분류하는 횟수가 많아져, 결과적으로 사기 거래는 덜 놓치게 된다. 
- 그러나 threshold를 너무 많이 낮추면, 과도하게 많은 거래를 사기로 분류할 수 있기 때문에 Precision이 떨어질 수 있다.

#### Q5. CTGAN / TVAE와 SMOTE의 가장 큰 차이를 한 문장으로 설명하세요.

**답변:**  
- SMOTE는 소수 클래스 이웃 샘플 간의 단순 선형 보간으로 새로운 데이터를 합성한다.
- CTGAN / TVAE는 딥러닝 생성 모델을 통해 다차원 피처들의 복잡한 확률 분포와 비선형 관계, 범주형/연속형 혼합 분포를 심층 학습하여 현실적인 합성 데이터를 생성한다.

#### Q6. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만으로 사기 탐지 성능을 판단할 때의 문제를 설명하세요.

**답변:**
- 정상 거래 건수가 사기 거래 건수에 비해 매우 많으므로(클래스 불균형이 심하므로), 전체 거래를 모두 정상거래라고 예측하여도 Accuracy는 높게 나온다.
- 이 지표만으로 사기 탐지 성능을 판단한다면 정상 거래는 많이 맞출 수 있으나 정작 중요한 사기 탐지 성능(Recall 재현율)은 떨어질 수 있다.


#### Q7. 사기 거래를 놓치는 FN의 비용과 데이터 특성을 고려해 class_weight·Random Undersampling·SMOTE 중 우선 적용할 방법을 하나 선택하고, 선택 이유와 한계를 설명하세요.

**답변:**
- class_weight 선택: 데이터의 손실이나 왜곡 없이 사용 가능하면서도 Recall을 효과적으로 올릴 수 있기 때문에 우선적으로 채택했다. 
- 그러나 가중치를 지나치게 높일 경우 정상 거래를 사기 거래로 잘 못 판단하는 경우가 많아져 Precision이 급격히 떨어질 수 있다.

---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 불균형 데이터에서 Accuracy 하나만 보면 위험한 이유는 무엇인가요?
- 다수 클래스만 잘 맞춰도 Accuracy가 높아질 수 있기 때문이다.

2. Precision은 어떤 오류가 중요할 때 보는 지표인가요?
- 양성이라고 예측한 것 중에 실제 양성 비율

3. Recall은 어떤 오류가 중요할 때 보는 지표인가요?
- 실제 양성 중에서 모델이 정확히 양성으로 판별한 비율

4. class_weight, Undersampling, SMOTE는 각각 어떤 방식으로 불균형을 다루나요?
- class_weight: 소수 클래스 오류에 더 큰 가중치
- Undersampling: 다수 클래스 데이터 감소
- SMOTE: 소수 클래스 합성 데이터 증가

5. 리샘플링은 Train / Test 중 어디에만 적용해야 하나요?
- Train 데이터셋에만 적용

6. threshold를 낮추면 일반적으로 Recall과 Precision은 어떻게 변하나요?
- Recall은 높아지고 Precision은 낮아지는 경향이 있다.